# Workflow Patterns with LangGraph

Three patterns for coordinating LLM calls into a single system, each built end to end:

1. **Prompt chaining** — steps run in sequence, each consuming the last one's output.
2. **Routing** — a classifier sends each request down the matching branch.
3. **Parallelization** — independent steps run at once, then merge.

The final section combines them into a multi-agent request router.

In [ ]:
%%capture
%pip install langgraph langchain-groq python-dotenv

In [ ]:
from typing import TypedDict

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

In [ ]:
def print_workflow_info(workflow, app=None):
    """Print the nodes and edges of a LangGraph workflow, and optionally draw it."""
    print("WORKFLOW INFORMATION")
    print("====================")
    print(f"Nodes: {list(workflow.nodes)}")
    print(f"Edges: {workflow.edges}")

    if app:
        from IPython.display import Image, display
        # draw_mermaid_png() renders remotely and needs no system packages.
        try:
            display(Image(app.get_graph().draw_mermaid_png()))
        except Exception:
            print(app.get_graph().draw_mermaid())

Initialize the model used by every workflow below.

In [ ]:
# Load GROQ_API_KEY from a .env file (this folder or any parent).
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

## 1. Prompt Chaining

Decompose a task into a sequence of LLM calls, each step consuming the previous step's output.

**Use case:** a job application assistant — read a job description, write a resume summary from it, then a cover letter from that summary.

Shared state passed between the steps.

In [ ]:
class ChainState(TypedDict):
    job_description: str
    resume_summary: str
    cover_letter: str

Node 1 — summarize a resume against the job description.

In [ ]:
def generate_resume_summary(state: ChainState) -> ChainState:
    prompt = f"""
You're a resume assistant. Read the following job description and summarize the key qualifications and experience the ideal candidate should have, phrased as if from the perspective of a strong applicant's resume summary.

Job Description:
{state['job_description']}
"""

    response = llm.invoke(prompt)

    return {**state, "resume_summary": response.content}

Node 2 — write the cover letter from that summary.

In [ ]:
def generate_cover_letter(state: ChainState) -> ChainState:
    prompt = f"""
You're a cover letter writing assistant. Using the resume summary below, write a professional and personalized cover letter for the following job.

Resume Summary:
{state['resume_summary']}

Job Description:
{state['job_description']}
"""

    response = llm.invoke(prompt)

    return {**state, "cover_letter": response.content}

### Build the graph

In [ ]:
workflow = StateGraph(ChainState)
workflow

Register both nodes.

In [ ]:
workflow.add_node("generate_resume_summary", generate_resume_summary)
workflow.add_node("generate_cover_letter", generate_cover_letter)

In [ ]:
workflow.set_entry_point("generate_resume_summary")

In [ ]:
workflow.add_edge("generate_resume_summary", "generate_cover_letter")

In [ ]:
workflow.set_finish_point("generate_cover_letter")

In [ ]:
print_workflow_info(workflow)

Compile into a runnable app.

In [ ]:
app = workflow.compile()

In [ ]:
from IPython.display import Image, display

# draw_mermaid_png() renders remotely and needs no system packages;
# fall back to the text definition if it is unavailable.
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())

Run it on a job description.

In [ ]:
input_state = {
        "job_description": "We are looking for a data scientist with experience in machine learning, NLP, and Python. Prior work with large datasets and experience deploying models into production is required."
}

result = app.invoke(input_state)

In [ ]:
result['resume_summary']

## 2. Routing

One node classifies the incoming request and dispatches it to the branch that handles that intent — a switchboard in front of specialized handlers.

**Use case:** decide whether a request is a summarize or a translate task, then route it.

Shared state.

In [ ]:
class RouterState(TypedDict):
    user_input: str
    task_type: str
    output: str

Classification schema, bound to the model as a tool so the label comes back structured.

In [ ]:
class Router(BaseModel):
    role: str = Field(..., description="Decide whether the user wants to summarize a passage  ouput 'summarize'  or translate text into French oupput translate.")
llm_router=llm.bind_tools([Router])

In [ ]:
response=llm_router.invoke("summarize this I love the sun its so warm")

In [ ]:
def router_node(state: RouterState) -> RouterState:
    routing_prompt = f"""
    Classify whether the user wants to 'summarize' a passage or 'translate' text into French.

    User Input: "{state['user_input']}"
    """

    try:
        response = llm_router.invoke(routing_prompt)
    except Exception:
        # Some hosted models emit a malformed tool call and the API rejects it.
        # A classification failure should not take the whole graph down.
        return {**state, "task_type": "summarize"}

    if response.tool_calls:
        label = response.tool_calls[0]["args"].get("role", "")
        if label in ("summarize", "translate"):
            return {**state, "task_type": label}   # becomes the next node's name
    return {**state, "task_type": "summarize"}

In [ ]:
def router(state: RouterState) -> str:
    return state['task_type']

Summarize branch.

In [ ]:
def summarize_node(state: RouterState) -> RouterState:
    prompt = f"Please summarize the following passage:\n\n{state['user_input']}"
    response = llm.invoke(prompt)
    
    return {**state, "task_type": "summarize", "output": response.content}

Translate branch.

In [ ]:
def translate_node(state: RouterState) -> RouterState:
    prompt = f"Translate the following text to French:\n\n{state['user_input']}"
    response = llm.invoke(prompt)

    return {**state, "task_type": "translate", "output": response.content}

In [ ]:
workflow = StateGraph(RouterState)


In [ ]:
workflow.add_node("router", router_node)
workflow.add_node("summarize", summarize_node)
workflow.add_node("translate", translate_node)

In [ ]:
workflow.set_entry_point("router")

Conditional edges map each `task_type` to its node.

In [ ]:
workflow.add_conditional_edges("router", router, {
    "summarize": "summarize",
    "translate": "translate"
})

In [ ]:
workflow.set_finish_point("summarize")
workflow.set_finish_point("translate")

In [ ]:
app = workflow.compile()


In [ ]:
from IPython.display import Image, display

# draw_mermaid_png() renders remotely and needs no system packages;
# fall back to the text definition if it is unavailable.
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())

Only `user_input` is supplied; the router fills in the rest.

In [ ]:
input_text = {
        "user_input": "Can you translate this sentence: I love programming?"
    }

result = app.invoke(input_text)

In [ ]:
print(result[ 'output'])
print(result['task_type'])

Second request, taking the other branch.

In [ ]:
input_text = {
        "user_input": "Can you summarize this sentence: I love programming so much it is the best thing ever. All I want to do is programming?"
    }

result = app.invoke(input_text)

In [ ]:
print(result[ 'output'])
print(result['task_type'])

## 3. Parallelization

Run independent steps concurrently instead of sequentially, then merge their results.

**Use case:** translate one sentence into French, Spanish and Japanese at the same time, then combine the three outputs.

Shared state — one field per translation plus the merged result.

In [ ]:
class State(TypedDict):
    text: str
    french: str
    spanish: str
    japanese: str
    combined_output: str

French node.

In [ ]:
def translate_french(state: State) -> dict:
    response = llm.invoke(f"Translate the following text to French:\n\n{state['text']}")
    return {"french": response.content.strip()}

Spanish node.

In [ ]:
def translate_spanish(state: State) -> dict:
    response = llm.invoke(f"Translate the following text to Spanish:\n\n{state['text']}")
    return {"spanish": response.content.strip()}

Japanese node.

In [ ]:
def translate_japanese(state: State) -> dict:
    response = llm.invoke(f"Translate the following text to Japanese:\n\n{state['text']}")
    return {"japanese": response.content.strip()}

Aggregator — merges the three translations once they finish.

In [ ]:
def aggregator(state: State) -> dict:
    combined = f"Original Text: {state['text']}\n\n"
    combined += f"French: {state['french']}\n\n"
    combined += f"Spanish: {state['spanish']}\n\n"
    combined += f"Japanese: {state['japanese']}\n"
    return {"combined_output": combined}

In [ ]:
graph = StateGraph(State)

In [ ]:
graph.add_node("translate_french", translate_french)
graph.add_node("translate_spanish", translate_spanish)
graph.add_node("translate_japanese", translate_japanese)
graph.add_node("aggregator", aggregator)

Fan out: all three translation nodes start from `START`, so they run in parallel.

In [ ]:
# Connect parallel nodes from START
graph.add_edge(START, "translate_french")
graph.add_edge(START, "translate_spanish")
graph.add_edge(START, "translate_japanese")

Fan in: each translation feeds the aggregator.

In [ ]:
# Connect all translation nodes to the aggregator
graph.add_edge("translate_french", "aggregator")
graph.add_edge("translate_spanish", "aggregator")
graph.add_edge("translate_japanese", "aggregator")

In [ ]:
# Final node
graph.add_edge("aggregator", END)

In [ ]:
# Compile the graph
app = graph.compile()

Run it.

In [ ]:
input_text = {
        "text": "Good morning! I hope you have a wonderful day."
}

result = app.invoke(input_text)

In [ ]:
result

## Multi-Agent Routing System

The routing pattern scaled up: a classifier dispatches each request to one of four specialized handlers — ride hailing, restaurant orders, groceries, or a fallback.

In [ ]:
class RouterState(TypedDict):
    user_input: str
    task_type: str
    output: str

class Router(BaseModel):
    role: str = Field(
        ..., 
        description="Classify the user request. Return exactly one of: 'ride_hailing_call', 'restaurant_order', 'groceries' and if you do not know output 'default_handler'"
    )

llm_router = llm.bind_tools([Router])

### Router logic

Classify the request; fall back to `default_handler` when the model returns no tool call.

In [ ]:
VALID_ROUTES = {"ride_hailing_call", "restaurant_order", "groceries", "default_handler"}


def router_node(state: RouterState) -> RouterState:
    try:
        response = llm_router.invoke(state["user_input"])
    except Exception:
        # A request the model cannot classify sometimes comes back as a malformed
        # tool call, which the API rejects. That is exactly what default_handler
        # is for, so route there rather than raising.
        return {**state, "task_type": "default_handler"}

    if response.tool_calls:
        label = response.tool_calls[0]["args"].get("role", "")
        if label in VALID_ROUTES:
            return {**state, "task_type": label}
    return {**state, "task_type": "default_handler"}


def router(state: RouterState) -> str:
    return state["task_type"]

In [ ]:
def ride_hailing_node(state: RouterState) -> RouterState:
    """
    Processes ride hailing requests by extracting pickup/dropoff locations and preferences
    """
    prompt = f"""
    You are a ride hailing assistant. Based on the user's request, extract and organize the following information:
    
    - Pickup location
    - Destination/dropoff location  
    - Preferred ride type (if mentioned)
    - Any special requirements
    - Estimated timing preferences
    
    User Request: "{state['user_input']}"
    
    Provide a clear summary of the ride request with all available details.
    """
    
    response = llm.invoke(prompt)
    
    return {
        **state, 
        "task_type": "ride_hailing_call", 
        "output": response.content.strip()
    }

def restaurant_order_node(state: RouterState) -> RouterState:
    """
    Processes restaurant orders by organizing menu items, quantities, and preferences
    """
    prompt = f"""
    You are a restaurant ordering assistant. Based on the user's request, organize the following information:
    
    - Menu items requested
    - Quantities for each item
    - Special modifications or dietary restrictions
    - Delivery or pickup preference
    - Any timing requirements
    
    User Request: "{state['user_input']}"
    
    Provide a clear, organized summary of the restaurant order with all details.
    """
    
    response = llm.invoke(prompt)
    
    return {
        **state, 
        "task_type": "restaurant_order", 
        "output": response.content.strip()
    }

def groceries_node(state: RouterState) -> RouterState:
    """
    Processes grocery delivery requests with driver pickup service
    """
    prompt = f"""
    You are a grocery delivery assistant for a service where our drivers pick up groceries for customers.
    
    Based on the user's request, organize the following information:
    
    Shopping List:
    - List of grocery items needed
    - Quantities or amounts for each item
    - Brand preferences (if mentioned)
    - Any dietary restrictions or organic preferences
    
    Store Information:
    - Preferred store or location
    - Budget considerations
    - Special instructions for finding items
    
    Delivery Details:
    - Delivery address (if provided)
    - Preferred delivery time window
    - Any special delivery instructions
    - Contact information for driver coordination
    
    Driver Instructions:
    - Substitution preferences (if item unavailable)
    - How to handle out-of-stock items
    - Any items requiring special handling (fragile, cold items)
    - Payment method (if mentioned)
    
    User Request: "{state['user_input']}"
    
    Provide a comprehensive delivery order summary that our driver can use to efficiently shop and deliver groceries. 
    Include estimated pickup time and any special notes for the shopping trip.
    
    Format the response as a clear, organized delivery order that includes all necessary details for our driver service.
    """
    
    response = llm.invoke(prompt)
    
    return {
        **state, 
        "task_type": "groceries", 
        "output": response.content.strip()
    }
def default_handler_node(state: RouterState) -> RouterState:
    prompt = f"""
    I couldn't classify your request into a specific category. 
    Let me provide general assistance for: "{state['user_input']}"
    
    I can help you with:
    - Ride hailing services
    -  Restaurant orders  
    -  Grocery shopping
    
    Please rephrase your request to match one of these services, or if you need assistance with something else, I will connect you with our customer support team who can provide personalized help.
    
    Would you like me to:
    1. Help you rephrase your request for one of our services
    2. Connect you with customer support for additional assistance
    """
    response = llm.invoke(prompt)
    return {**state, "task_type": "default_handler", "output": response.content.strip()}


### Assemble the workflow

In [ ]:
workflow = StateGraph(RouterState)
# Add all nodes
workflow.add_node("router", router_node)
workflow.add_node("ride_hailing_call", ride_hailing_node)
workflow.add_node("restaurant_order", restaurant_order_node)
workflow.add_node("groceries", groceries_node)
workflow.add_node("default_handler", default_handler_node)

# Set entry point
workflow.set_entry_point("router")

# Add conditional routing
workflow.add_conditional_edges("router", router, {
    "groceries": "groceries", 
    "restaurant_order": "restaurant_order",
    "ride_hailing_call": "ride_hailing_call",
    "default_handler": "default_handler"
})

# Set finish points
workflow.set_finish_point("ride_hailing_call")
workflow.set_finish_point("restaurant_order")
workflow.set_finish_point("groceries")
workflow.set_finish_point("default_handler")

# Compile the application
app = workflow.compile()

Test across all four intents.

In [ ]:
test_cases = [
    {"user_input": "I need a ride from downtown to the airport at 3pm"},
    {"user_input": "I want to order 2 large pepperoni pizzas for delivery"},
    {"user_input": "I need milk, bread, eggs, and vegetables for the week"},
    {"user_input": "What's the weather like today?"},  # Default/unclassified example
]

for i, test_input in enumerate(test_cases, 1):
    result=app.invoke(test_input)


    print(f"question: {test_input['user_input']}\n")
    print(f"task_type {result['task_type']}\n")
    print(f"output: {result['output']}\n")
    print('-----------------------------------')

## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)